In [52]:
# 配置环境变量
from dotenv import load_dotenv
load_dotenv()
import os
import json
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    raise ValueError(
        "\n请先在 .env 文件中设置有效的 GROQ_API_KEY\n"
        "访问 https://console.groq.com/keys 获取免费密钥"
    )

from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY
)

from pydantic import BaseModel, Field
from enum import Enum
from typing import List, Optional, TypeVar, Type
# 泛型类型
T = TypeVar("T", bound=BaseModel)

In [55]:
def safe_structured_output(prompt: str, output_class: Type[T]) -> T:
    """
    安全的结构化输出函数

    先尝试 with_structured_output，失败则使用 JSON 解析 fallback
    """
    # 尝试使用 with_structured_output
    try:
        structured_llm = create_safe_structured_llm(output_class)
        result = structured_llm.invoke(prompt)
        return result
    except Exception as e:
        print(f"  ⚠️ with_structured_output 失败: {e}")
        print("  📝 使用 JSON 解析 fallback...")

    # Fallback: 手动 JSON 解析
    json_prompt = f"""{prompt}

请严格按照以下JSON格式返回（不要添加任何其他内容）：
{json.dumps(output_class.model_json_schema().get('properties', {}), indent=2, ensure_ascii=False)}

只返回JSON，不要其他文字。"""

    response = model.invoke([HumanMessage(content=json_prompt)])
    content = response.content.strip()

    # 清理 Markdown 格式
    if "```json" in content:
        content = content.split("```json")[1].split("```")[0]
    elif "```" in content:
        content = content.split("```")[1].split("```")[0]

    try:
        data = json.loads(content.strip())
        return output_class.model_validate(data)
    except Exception as e2:
        print(f"  ❌ JSON 解析也失败: {e2}")
        raise ValueError(f"无法解析结构化输出: {e2}")

In [56]:
def create_safe_structured_llm(output_class):
    """创建带 fallback 的结构化输出 LLM"""
    base_llm = model.with_structured_output(output_class)

    class SafeStructuredLLM:
        def invoke(self, prompt):
            try:
                return base_llm.invoke(prompt)
            except Exception as e:
                print(f"  ⚠️ 结构化输出失败，使用 fallback: {e}")
                return safe_structured_output(prompt, output_class)

    return SafeStructuredLLM()


# ============================================================================
# 示例 1：基础结构化输出
# ============================================================================

In [57]:
class Sentiment(str, Enum):
    """情感"""
    POSITIVE = "正面"
    NEUTRAL = "中性"
    NEGATIVE = "负面"

In [58]:
class Review(BaseModel):
    """评论"""
    product: str = Field(description="产品名称")
    rating: int = Field(description="评分 1-5")
    sentiment: Sentiment = Field(description="情感倾向")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")

In [59]:
def example():
    structured_llm = create_safe_structured_llm(Review)
    review_text = """
    这款 iPhone 15 Pro 真的很不错！摄像头非常强大，夜拍效果惊艳。
    钛金属边框手感也很好。但是价格有点贵，而且没有充电器。
    总体来说还是值得购买的，我给 4 分。
    """
    print(f"\n评论内容：\n{review_text}")
    res = structured_llm.invoke(f"分析以下产品评论：\n{review_text}")

    print(f"\n分析结果：")
    print(f"产品：{res.product}")
    print(f"评分：{res.rating} / 5")
    print(f"情感：{res.sentiment.value}")
    print(f"优点：{'，'.join(res.pros)}")
    print(f"缺点：{'，'.join(res.cons)}")



In [60]:
def main():
    print("\n"+"="*70)
    print("Structured_Output")
    print("="*70)

    example()

In [61]:
if __name__ == "__main__":
    main()


Structured_Output

评论内容：

    这款 iPhone 15 Pro 真的很不错！摄像头非常强大，夜拍效果惊艳。
    钛金属边框手感也很好。但是价格有点贵，而且没有充电器。
    总体来说还是值得购买的，我给 4 分。
    

分析结果：
产品：iPhone 15 Pro
评分：4 / 5
情感：正面
优点：摄像头强大，夜拍效果惊艳，钛金属边框手感好
缺点：价格贵，没有充电器
